# 04 · Limpieza de planos de siembra

**Entran:** los 294 Excel originales semanales. **Sale:** `planos_siembra_limpio.parquet`.

Una fila es una línea del plano de una fecha, no un evento de siembra nuevo. El historial de observaciones se conserva dentro de **una sola tabla**, sin CSV por semana. Se mantienen las dos ediciones originales de 2026-S32 con una marca de selección para no sumar dos veces el mismo plano.

## 1. Imports y rutas

In [1]:
from pathlib import Path
import sys
import re
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos").is_dir() and (p / "Notebooks").is_dir())
from datetime import date
import unicodedata
from pathlib import PurePosixPath
from zipfile import ZipFile
from lxml import etree

def texto(valor):
    if pd.isna(valor):
        return ""
    s = unicodedata.normalize("NFKD", str(valor).replace("\xa0", " "))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().upper()

def normalizar(serie):
    return (serie.astype("string").str.normalize("NFKD")
            .str.replace("[\u0300-\u036f]", "", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
            .replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "<NA>": pd.NA}))

def finca(serie):
    return normalizar(serie).replace({"GF": "GAITANA", "LA GAITANA": "GAITANA", "¡": "GAITANA",
                                     "AR": "ARABELLA", "LA CABANA": "CABANA", "FINCA LA CABANA": "CABANA"})

def fechas(serie):
    """Conservar calendario de Excel: fechas ya leídas o seriales 1900, no nanosegundos 1970."""
    salida = pd.to_datetime(serie, errors="coerce", format="mixed")
    numeros = pd.to_numeric(serie.astype("string"), errors="coerce")
    serial = numeros.between(20000, 80000)
    salida.loc[serial] = pd.to_datetime(numeros[serial], unit="D", origin="1899-12-30")
    return salida

def semana_operativa(anio, semana):
    """Lunes representativo. Toda etiqueta 53 se convierte en 01 del año siguiente."""
    y = pd.to_numeric(anio, errors="coerce").astype("Int64")
    w = pd.to_numeric(semana, errors="coerce").astype("Int64")
    y = y + w.eq(53).fillna(False).astype(int)
    w = w.mask(w.eq(53), 1)
    etiquetas = y.astype("string") + "-" + w.astype("string")
    conversion = {}
    for etiqueta in etiquetas.dropna().unique():
        try:
            yy, ww = map(int, etiqueta.split("-"))
            conversion[etiqueta] = pd.Timestamp(date.fromisocalendar(yy, ww, 1)) if 1 <= ww <= 52 else pd.NaT
        except (ValueError, OverflowError):
            conversion[etiqueta] = pd.NaT
    return pd.to_datetime(etiquetas.map(conversion))

def leer_tabla_excel(libro, hoja, obligatorias):
    """Detectar el encabezado desplazado; devolver datos y número de fila Excel inicial."""
    inicio = pd.read_excel(libro, sheet_name=hoja, header=None, nrows=12)
    requeridas = {texto(c) for c in obligatorias}
    candidatos = [i for i, fila in inicio.iterrows() if requeridas <= {texto(x) for x in fila}]
    if len(candidatos) != 1:
        raise ValueError(f"Encabezado ambiguo/ausente en {hoja}: {obligatorias}")
    encabezado = candidatos[0]
    d = pd.read_excel(libro, sheet_name=hoja, header=encabezado)
    d.columns = [texto(c) for c in d.columns]
    return d, encabezado + 2

SALIDA = ROOT / "Datos_analiticos_proyecto"
SALIDA.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 35)
print("Fuentes originales:", ROOT / "Datos")
print("Salida oficial:", SALIDA)

ENTRADA = ROOT / "Datos/Plano de Siembras"
ARCHIVO_SALIDA = SALIDA / "planos_siembra_limpio.parquet"
archivos = sorted(p for p in ENTRADA.glob("*/*.xlsx") if re.fullmatch(r"20\d{2}",p.parent.name))
inventario = pd.DataFrame({"archivo":[str(p.relative_to(ENTRADA)) for p in archivos],
                          "anio":[int(p.parent.name) for p in archivos],
                          "semana":[int(re.search(r"SEM\s*(\d{1,2})",texto(p.stem)).group(1)) for p in archivos]})
inventario["actualizado"] = inventario.archivo.str.upper().str.contains("ACTUALIZADO")
inventario = inventario.sort_values(["anio","semana","actualizado","archivo"])
inventario["edicion_elegida"] = ~inventario.duplicated(["anio","semana"],keep="last")
display(inventario.groupby("anio").size().to_frame("libros"))
display(inventario[inventario.duplicated(["anio","semana"],keep=False)])


Fuentes originales: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos
Salida oficial: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto


,libros
anio,
2021,52
2022,52
2023,52
2024,52
2025,52
2026,34


,archivo,anio,semana,actualizado,edicion_elegida
292,2026\Plano de Siembras Sem 32.xlsx,2026,32,False,False
291,2026\Plano de Siembras Sem 32 actualizado.xlsx,2026,32,True,True


## 2. Encabezado y grano

Se identifica la fila que contiene FINCA, PRODUCTO, VARIEDAD y PLANTAS; otras hojas son reportes. El original 2022-S02 no contiene Datos y se informa como faltante, sin fabricar observaciones.

In [2]:
libro = pd.ExcelFile(archivos[0],engine="calamine")
ejemplo, fila_inicial = leer_tabla_excel(libro,"Datos",["FINCA","PRODUCTO","VARIEDAD","PLANTAS"])
display(ejemplo.head())
print("Columnas:",ejemplo.columns.tolist())
libro.close()
del ejemplo


,NOMCIA,COD-FINCA,FINCA,COD-INVERNADERO,INVERNADERO,SECTOR,NAVE,CAMA,NRO CAMA,PRODUCTO,COLOR,VARIEDAD,TIPO DE FLOR,PLANTAS,AREA,FECHA_SIEMBRA,SEM_SIEMBRA,...,UNNAMED: 69,UNNAMED: 70,UNNAMED: 71,UNNAMED: 72,UNNAMED: 73,UNNAMED: 74,UNNAMED: 75,UNNAMED: 76,AREA DEL BLOQUE,UNNAMED: 78,UNNAMED: 79,UNNAMED: 80,BLOQUE.1,ORDEN,UNNAMED: 83,UNNAMED: 84,UNNAMED: 85
0,NaN,NaN,Arabella,NaN,Bloque 01,Amalia Rodriguez,NaN,57.0,NaN,Carnation,gold,Caroline Gold,NaN,1202.0,NaN,2020-04-08,15.0,...,NaN,NaN,Sustrato Gaitana,50-30-20,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Bloque 06,1,NaN,NaN,NaN
1,NaN,NaN,Arabella,NaN,Bloque 01,Amalia Rodriguez,NaN,55.0,NaN,Carnation,gold,Caroline Gold,NaN,1200.0,NaN,2020-04-08,15.0,...,NaN,NaN,Sustrato 90%,Cascarilla 90%,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Bloque 05,2,NaN,NaN,NaN
2,NaN,NaN,Arabella,NaN,Bloque 01,Amalia Rodriguez,NaN,35.0,NaN,Carnation,burgundy,Zurigo,NaN,434.0,NaN,2019-11-30,48.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Bloque 11,3,NaN,NaN,NaN
3,NaN,NaN,Arabella,NaN,Bloque 01,Amalia Rodriguez,NaN,11.0,NaN,Carnation,purple,Zafiro,NaN,1200.0,NaN,2020-04-03,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Bloque 02,4,NaN,NaN,NaN
4,NaN,NaN,Arabella,NaN,Bloque 01,Amalia Rodriguez,NaN,8.0,NaN,Carnation,purple,Zafiro,NaN,1200.0,NaN,2020-04-03,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,Bloque 21,5,NaN,NaN,NaN


Columnas: ['NOMCIA', 'COD-FINCA', 'FINCA', 'COD-INVERNADERO', 'INVERNADERO', 'SECTOR', 'NAVE', 'CAMA', 'NRO CAMA', 'PRODUCTO', 'COLOR', 'VARIEDAD', 'TIPO DE FLOR', 'PLANTAS', 'AREA', 'FECHA_SIEMBRA', 'SEM_SIEMBRA', 'ANOSIEM', 'CURVA', 'FECHA PRODUCCION', 'SEM_PRODUCCION', 'ANOPROD', 'SEM COLOCACION CORTINAS', 'CICLO EN DIAS', 'COMPROMETIDO', 'COD', 'CD VALVULA', 'SUSTRATO', 'TABLA', 'PERIODO-PROY', 'DENSIDAD-M2', 'CODSIEM', 'NOMSIEM', 'SUPERVISOR', 'NORMA DESPUNTE', 'DESPUNTE/REPASO', 'PROVEEDOR', 'EDAD', 'EDAD INF', 'ESTADO', 'PROGIBB', 'CALCIBOR', 'RAJADO', 'BAJO PESO', 'APLICACION A CORTES', 'AREA BRUTA', 'TIPO DE SUSTRATO', 'AREA BLOQUE', 'SUPERVISOR.1', 'VARIEDAD.1', 'BLOQUE', 'CAMA.1', 'FECHA DE SIEMBRA', 'PLANTAS.1', 'SUSTRATO.1', 'UNNAMED: 55', 'REFERENCIA', 'REFERENCIA.1', 'REFERENCIA.2', 'REFERENCIA.3', 'PLANTAS.2', 'CAMA.2', 'SUSTRATO.2', 'UNNAMED: 63', 'UNNAMED: 64', 'UNNAMED: 65', 'UNNAMED: 66', 'UNNAMED: 67', 'UNNAMED: 68', 'UNNAMED: 69', 'UNNAMED: 70', 'UNNAMED: 71', 'UN

## 3. Lectura, selección y normalización

Plantas y área son medidas diferentes y se conservan separadas. Se mantiene edad reportada y fecha de siembra; la diferencia entre ambas se diagnostica, no se corrige silenciosamente.

In [3]:
aliases = {"FINCA":"finca","INVERNADERO":"invernadero","CAMA":"cama","PRODUCTO":"producto",
           "COLOR":"color","VARIEDAD":"variedad","PLANTAS":"plantas","AREA":"area",
           "FECHA_SIEMBRA":"fecha_siembra","SEM_SIEMBRA":"semana_siembra_fuente","ANOSIEM":"anio_siembra_fuente",
           "EDAD":"edad_reportada","CURVA":"curva_fuente","CODSIEM":"codigo_siembra","ESTADO":"estado",
           "SUSTRATO":"sustrato","DENSIDAD-M2":"densidad"}
partes, controles = [], []
for i,fila in inventario.iterrows():
    archivo = ENTRADA / fila.archivo
    libro = pd.ExcelFile(archivo,engine="calamine")
    if "Datos" not in libro.sheet_names:
        controles.append({"archivo":fila.archivo,"estado":"SIN HOJA DATOS","filas":0})
        print("Sin hoja Datos:",fila.archivo,flush=True)
        libro.close()
        continue
    df, inicio = leer_tabla_excel(libro,"Datos",["FINCA","PRODUCTO","VARIEDAD","PLANTAS"])
    df = df[[c for c in df if c in aliases]].rename(columns=aliases)
    df = df.reindex(columns=list(aliases.values()))
    df["fila_excel"] = np.arange(inicio,inicio+len(df))
    # No incluir anotaciones laterales o filas completamente vacías fuera del detalle.
    df = df.loc[df[["finca","producto","variedad","plantas","fecha_siembra"]].notna().any(axis=1)].copy()
    df["finca"] = finca(df.finca)
    for col in ["producto","color","variedad","invernadero","cama"]:
        df[col] = normalizar(df[col])
    df["fecha_siembra"] = fechas(df.fecha_siembra).dt.normalize()
    for col in ["plantas","area","edad_reportada","densidad","semana_siembra_fuente","anio_siembra_fuente"]:
        df[col] = pd.to_numeric(df[col],errors="coerce")
    for col in ["curva_fuente","codigo_siembra","estado","sustrato"]:
        df[col] = df[col].astype("string")
    df["semana_inicio"] = semana_operativa(pd.Series(fila.anio,index=df.index),pd.Series(fila.semana,index=df.index))
    df["anio_plano_fuente"],df["semana_plano_fuente"] = fila.anio,fila.semana
    df["edicion_elegida"] = bool(fila.edicion_elegida)
    df["edad_calculada"] = np.floor((df.semana_inicio-df.fecha_siembra).dt.days/7)
    df["edad_inconsistente"] = (df.edad_reportada-df.edad_calculada).abs().gt(2)
    df["archivo_origen"] = str(archivo.relative_to(ROOT)).replace("\\","/")
    df["hoja_origen"] = "Datos"
    controles.append({"archivo":fila.archivo,"estado":"OK","filas":len(df)})
    partes.append(df)
    libro.close()
    if len(controles)%20==0: print(f"Leídos {len(controles)}/{len(inventario)} libros",flush=True)
planos = pd.concat(partes,ignore_index=True)
del partes, df
display(pd.DataFrame(controles).groupby("estado").agg(libros=("archivo","size"),filas=("filas","sum")))
display(pd.DataFrame(controles).query("estado != 'OK'"))


Leídos 20/294 libros


Leídos 40/294 libros


Sin hoja Datos: 2022\Plano de Siembras Sem 02.xlsx


Leídos 60/294 libros


Leídos 80/294 libros


...

Leídos 240/294 libros


Leídos 260/294 libros


Leídos 280/294 libros


,libros,filas
estado,,
OK,293,3252152
SIN HOJA DATOS,1,0


,archivo,estado,filas
53,2022\Plano de Siembras Sem 02.xlsx,SIN HOJA DATOS,0


## 4. Duplicados y cobertura

Un mismo identificador cama/siembra puede aparecer en segmentos distintos. Se marca la ambigüedad: no eliminar ni sumar ciegamente esos segmentos para estimar potencial. Una cantidad faltante no significa cama vacía.

In [4]:
llave_segmento = ["archivo_origen","finca","invernadero","cama","producto","color","variedad","fecha_siembra"]
planos["segmento_ambiguo"] = planos.duplicated(llave_segmento,keep=False)
planos["fila_utilizable"] = (planos[["finca","producto","color","variedad","fecha_siembra"]].notna().all(axis=1)
                            & planos.plantas.ge(0) & ~planos.segmento_ambiguo)
assert not planos.duplicated(["archivo_origen","fila_excel"]).any()
display(planos.groupby("finca",dropna=False).agg(filas=("fila_excel","size"),semanas=("semana_inicio","nunique"),
                  desde=("semana_inicio","min"),hasta=("semana_inicio","max"),ambiguas=("segmento_ambiguo","sum")))
display(planos[["plantas","area","fecha_siembra","edad_reportada"]].isna().sum().to_frame("faltantes"))
print("Edición elegida:",int(planos.edicion_elegida.sum()),"filas")


,filas,semanas,desde,hasta,ambiguas
finca,,,,,
ARABELLA,999863,292,2021-01-04,2026-08-10,16934
GAITANA,2251469,292,2021-01-04,2026-08-10,20325
GAITANA PPTO,648,1,2022-06-13,2022-06-13,2
<NA>,172,18,2022-02-21,2025-02-03,62


,faltantes
plantas,255
area,3197720
fecha_siembra,166
edad_reportada,19520


Edición elegida: 3240395 filas


## 5. Validar la edad con la fecha que realmente usa Excel

La fórmula original de EDAD usa `ROUND((AL1 - fecha_siembra)/7, 0)`. AL1 no necesariamente es el lunes del nombre del archivo. Compararlas como si fueran la misma fecha marcaba falsamente muchas edades como inconsistentes.

Leemos AL1 y comprobamos la fórmula en cada libro. Conservamos la edad al lunes (semanas completas), la reportada, su fecha de referencia y ambas banderas. No movemos fechas de siembra ni sumamos semanas arbitrarias. Si no reconocemos la fórmula, el registro queda pendiente.

**Importante:** AL1 es una fecha para calcular edad, no prueba cuándo llegó el plano ni cuándo se generó. Usar planos en un backtest seguirá requiriendo un supuesto de disponibilidad explícito.


In [5]:
from zipfile import ZipFile
from lxml import etree
from pathlib import PurePosixPath

def referencia_edad_del_libro(ruta):
    """Leer sólo cabecera y fórmula de edad del Excel; nunca recalcularlo ni guardarlo."""
    ns = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    rel_ns = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    with ZipFile(ruta) as z:
        wb = etree.fromstring(z.read("xl/workbook.xml"))
        hoja = next(h for h in wb.find(ns+"sheets") if h.get("name")=="Datos")
        relaciones = etree.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        destino = next(r.get("Target") for r in relaciones if r.get("Id")==hoja.get(rel_ns+"id"))
        destino = destino.lstrip("/") if destino.startswith("/") else str(PurePosixPath("xl")/destino)
        celdas = {}
        with z.open(destino) as flujo:
            for _, fila in etree.iterparse(flujo,events=("end",),tag=ns+"row"):
                for celda in fila:
                    if celda.get("r") in ["AL1","AL2","AL3"]:
                        celdas[celda.get("r")] = (celda.findtext(ns+"v"),celda.findtext(ns+"f"),celda.get("t"))
                if int(fila.get("r"))>=3:break
        valor,_,tipo = celdas.get("AL1",(None,None,None))
        formula = celdas.get("AL3",(None,None,None))[1] or ""
        formula_reconocida = "$AL$1" in formula and "ROUND(" in formula.upper()
        fecha = pd.NaT
        if valor is not None and tipo not in ["s","str","b","e"]:
            propiedad = wb.find(ns+"workbookPr")
            origen = "1904-01-01" if propiedad is not None and propiedad.get("date1904") in ["1","true"] else "1899-12-30"
            fecha = pd.Timestamp(origen)+pd.Timedelta(days=float(valor))
        return fecha,formula_reconocida

referencias = []
for archivo in planos.archivo_origen.unique():
    fecha,formula = referencia_edad_del_libro(ROOT/archivo)
    referencias.append({"archivo_origen":archivo,"fecha_referencia_edad":fecha,
                        "formula_edad_reconocida":formula})
referencias = pd.DataFrame(referencias)
# Permite ejecutar esta etapa nuevamente sin crear columnas duplicadas.
planos = planos.drop(columns=["fecha_referencia_edad","formula_edad_reconocida",
                             "edad_diferente_al_lunes","edad_esperada_referencia"],errors="ignore")
planos = planos.merge(referencias,on="archivo_origen",how="left",validate="many_to_one")
planos["edad_diferente_al_lunes"] = (planos.edad_reportada-planos.edad_calculada).abs().gt(2)
edad_decimal = (planos.fecha_referencia_edad-planos.fecha_siembra).dt.days/7
# ROUND de Excel: medios alejándose de cero, no redondeo bancario de Python.
planos["edad_esperada_referencia"] = np.sign(edad_decimal)*np.floor(np.abs(edad_decimal)+.5)
verificable = planos.formula_edad_reconocida & planos.fecha_referencia_edad.notna()
planos["edad_inconsistente"] = ((planos.edad_reportada-planos.edad_esperada_referencia).abs().gt(.01)
                               | planos.edad_reportada.isna() | ~verificable)
assert not planos.duplicated(["archivo_origen","fila_excel"]).any()
display(referencias.groupby("formula_edad_reconocida").size().to_frame("libros"))
display(planos.groupby("finca",dropna=False).agg(
    filas=("fila_excel","size"),diferente_al_lunes=("edad_diferente_al_lunes","sum"),
    no_coincide_formula=("edad_inconsistente","sum")))
display(planos[["semana_inicio","fecha_referencia_edad","fecha_siembra",
                "edad_reportada","edad_calculada","edad_esperada_referencia"]].head())


,libros
formula_edad_reconocida,
True,293


,filas,diferente_al_lunes,no_coincide_formula
finca,,,
ARABELLA,999863,580999,6013
GAITANA,2251469,1093123,23835
GAITANA PPTO,648,496,0
<NA>,172,0,67


,semana_inicio,fecha_referencia_edad,fecha_siembra,edad_reportada,edad_calculada,edad_esperada_referencia
0,2021-01-04,2021-01-18,2020-04-08,41.0,38.0,41.0
1,2021-01-04,2021-01-18,2020-04-08,41.0,38.0,41.0
2,2021-01-04,2021-01-18,2019-11-30,59.0,57.0,59.0
3,2021-01-04,2021-01-18,2020-04-03,41.0,39.0,41.0
4,2021-01-04,2021-01-18,2020-04-03,41.0,39.0,41.0


## 6. Exportación única

No se exporta otro consolidado por año, cama o semana. La base analítica agregará las filas admisibles y mostrará cobertura. Fecha del plano no equivale a fecha de publicación certificada.

In [6]:
planos.to_parquet(ARCHIVO_SALIDA,index=False)
print("EXPORTADO:",ARCHIVO_SALIDA,planos.shape)
display(planos.head())


EXPORTADO: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto\planos_siembra_limpio.parquet (3252152, 32)


,finca,invernadero,cama,producto,color,variedad,plantas,area,fecha_siembra,semana_siembra_fuente,anio_siembra_fuente,edad_reportada,curva_fuente,codigo_siembra,estado,sustrato,densidad,fila_excel,semana_inicio,anio_plano_fuente,semana_plano_fuente,edicion_elegida,edad_calculada,edad_inconsistente,archivo_origen,hoja_origen,segmento_ambiguo,fila_utilizable,fecha_referencia_edad,formula_edad_reconocida,edad_diferente_al_lunes,edad_esperada_referencia
0,ARABELLA,BLOQUE 01,57.0,CARNATION,GOLD,CAROLINE GOLD,1202.0,NaN,2020-04-08,15.0,2020.0,41.0,<NA>,<NA>,<NA>,Sustrato 90%,NaN,3,2021-01-04,2021,1,True,38.0,False,Datos/Plano de Siembras/2021/Plano de Siembras...,Datos,False,True,2021-01-18,True,True,41.0
1,ARABELLA,BLOQUE 01,55.0,CARNATION,GOLD,CAROLINE GOLD,1200.0,NaN,2020-04-08,15.0,2020.0,41.0,<NA>,<NA>,<NA>,Sustrato 90%,NaN,4,2021-01-04,2021,1,True,38.0,False,Datos/Plano de Siembras/2021/Plano de Siembras...,Datos,False,True,2021-01-18,True,True,41.0
2,ARABELLA,BLOQUE 01,35.0,CARNATION,BURGUNDY,ZURIGO,434.0,NaN,2019-11-30,48.0,2019.0,59.0,<NA>,<NA>,<NA>,Sustrato 90%,NaN,5,2021-01-04,2021,1,True,57.0,False,Datos/Plano de Siembras/2021/Plano de Siembras...,Datos,False,True,2021-01-18,True,False,59.0
3,ARABELLA,BLOQUE 01,11.0,CARNATION,PURPLE,ZAFIRO,1200.0,NaN,2020-04-03,14.0,2020.0,41.0,<NA>,<NA>,<NA>,Sustrato 90%,NaN,6,2021-01-04,2021,1,True,39.0,False,Datos/Plano de Siembras/2021/Plano de Siembras...,Datos,False,True,2021-01-18,True,False,41.0
4,ARABELLA,BLOQUE 01,8.0,CARNATION,PURPLE,ZAFIRO,1200.0,NaN,2020-04-03,14.0,2020.0,41.0,<NA>,<NA>,<NA>,Sustrato 90%,NaN,7,2021-01-04,2021,1,True,39.0,False,Datos/Plano de Siembras/2021/Plano de Siembras...,Datos,False,True,2021-01-18,True,False,41.0
